<a href="https://colab.research.google.com/github/christinengalle19-collab/Assignement2New/blob/main/Assignment_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##For this assignment, I selected a text dataset from Project Gutenberg. It provides public domain books that are legally accessible and easy to preprocess. The dataset is lightweight, suitable for training a simple generative model, and allows clear evaluation of text coherence, vocabulary patterns, and stylistic reproduction.

In [ ]:
!pip install transformers torch
!pip install requests
!pip install torch
!pip install transformers


In [ ]:
import numpy as np

import requests
import re
from transformers import pipeline, set_seed
import torch
import tensorflow as tf
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Embedding

In [ ]:
#load the text from gutenberg project
url = "https://www.gutenberg.org/files/11/11-0.txt"
response = requests.get(url)
text = response.text
print(text[:500])

*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The


In [ ]:
#clean the text byremoving header and footer
text = re.sub(r'\[.*?\]', '', text)
text = text.strip()
print(text[:500])


*** START OF THE PROJECT GUTENBERG EBOOK 11 ***






Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s


In [ ]:
#save the text in a file
with open("shakespeare.txt", "w", encoding="utf-8") as file:
    file.write(text)
    print("Text saved to shakespeare.txt")

Text saved to shakespeare.txt


In [ ]:
## Load the pre-trained model and tokenizer
#Next, we'll load a pre-trained GPT-2 model and its associated tokenizer. The tokenizer is responsible for converting text into tokens that the model can understand.

model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
#tokenizing the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_index = len(tokenizer.word_index) + 1
print(total_index)

3066


In [ ]:
# Analyze the vocabulary
#top 20 frequent words
sorted_words = sorted(tokenizer.word_counts.items(), key=lambda x: x[1], reverse=True)
top_words = sorted_words[:20]
print("Top 20 frequent words:")
for word, count in top_words:
    print(f"{word}: {count}")
    print("\n")


Top 20 frequent words:
the: 1623


”: 1040


and: 795


to: 719


a: 621


she: 535


of: 502


it: 494


said: 459


alice: 385


in: 361


was: 356


you: 319


i: 273


that: 264


as: 254


her: 248


at: 206


on: 192


with: 179




In [ ]:
#split for training
sequences = []
for line in text.split('\n'):
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        n_gram_sequence = tokens[:i+1]
sequences = tokenizer.texts_to_sequences([text])
sequences = np.array(sequences)
max_length = max([len(seq) for seq in sequences])
sequences = pad_sequences(sequences, maxlen=max_length, padding='pre')
x = sequences[:, :-1]
y = sequences[:, -1]
y = to_categorical(y, num_classes=total_index)
print(x.shape)
print(y.shape)


(1, 27763)
(1, 3066)


In [ ]:
#Modele LSTM
model = Sequential()
model.add(Embedding(total_index, 10, input_length=max_length-1))
model.add(LSTM(50))
model.add(Dense(total_index, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x, y, epochs=100, verbose=1)


Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 186s 186s/step - accuracy: 0.0000e+00 - loss: 8.0278
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 102s 102s/step - accuracy: 1.0000 - loss: 8.0214
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 96s 96s/step - accuracy: 1.0000 - loss: 8.0145
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 110s 110s/step - accuracy: 1.0000 - loss: 8.0062
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 119s 119s/step - accuracy: 1.0000 - loss: 7.9955
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 98s 98s/step - accuracy: 1.0000 - loss: 7.9805
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 91s 91s/step - accuracy: 1.0000 - loss: 7.9580
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 102s 102s/step - accuracy: 1.0000 - loss: 7.9195
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 130s 130s/step - accuracy: 1.0000 - loss: 7.8340
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 93s 93s/step - accuracy: 1.0000 - loss: 7.4828
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 97s 97s/step - accuracy: 1.0000 - loss: 6.8515
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 100s 100s/step - acc

In [ ]:
## Define a function for text generation
#This function will be our main tool for generating text. It takes a prompt and several parameters that control the generation process.


def generate_text(prompt, max_length=100, temperature=0.7, num_return_sequences=1):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    output = model.generate(
        input_ids,
        max_length=max_length,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

